# 세그먼트 분리 — 4분할 × 3모델 앙상블 (케글)
**골격:** 기획 전달본(4분할·공유fold·세그먼트별 3모델·OOF·기본파라미터). **목표=측정**(0.74219 갱신 아님). 안전판=현 best 0.74219.
**원칙:** OOF만(in-sample 금지)·공유 fold·구조만(피처/튜닝 X)·기존 lgb/cat/xgb 재사용.
**게이트 2종:** ① 세그먼트 vs **같은 base 비분리 글로벌**(구조 효과 격리) ② 세그먼트 vs **0.74219 운영 best**(이길 수 있나).
**smoke→풀런:** `DRYRUN=True`(서브샘플)로 정합·assert 통과 후 `False`.

## 0. 설정

In [1]:
import os, re, warnings, time
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")
DRYRUN=False                       # smoke 먼저 → False 풀런
SEEDS=(42,) if DRYRUN else (42,7,2024)
NE_FULL=1500 if not DRYRUN else 200; NE_SHALLOW=400 if not DRYRUN else 100
_IDX={}
def _bi():
    import os
    for root in ["/kaggle/input","/kaggle/working","."]:
        if not os.path.isdir(root): continue
        for dp,dn,fn in os.walk(root):
            dn[:]=[d for d in dn if d not in ("catboost_info",".virtual_documents",".git","__pycache__")]
            for f in fn:
                if f.endswith(".csv") and f not in _IDX: _IDX[f]=os.path.join(dp,f)
_bi()
def P(n): 
    assert n in _IDX, f"{n} 없음"; return _IDX[n]
train=pd.read_csv(P("train.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"
y=train[TARGET].astype(int).values; N=len(train)
def runr(x): return rankdata(x)/len(x)
if DRYRUN:   # 서브샘플(세그먼트 비율 유지 위해 층화 비슷하게 — 단순 랜덤 12k)
    idx=np.random.default_rng(42).choice(N,min(12000,N),replace=False); idx.sort()
    train=train.iloc[idx].reset_index(drop=True); y=y[idx]; N=len(train)
print(f"train {train.shape} | base_rate={y.mean():.4f} | DRYRUN={DRYRUN}")

train (256351, 69) | base_rate=0.2583 | DRYRUN=False


## 1. CELL1 — 세그먼트 4분할 배정 (train/test 동일, disjoint, 미분류 0)

In [2]:
def assign_segment(df):
    di =df["시술 유형"]=="DI"; ivf=df["시술 유형"]=="IVF"
    now=df["배아 생성 주요 이유"].astype(str).str.contains("현재 시술용",na=False)
    tx =pd.to_numeric(df["이식된 배아 수"],errors="coerce").fillna(0)
    seg=pd.Series("S2",index=df.index)          # 기본 S2 (IVF·현재시술용·이식>0)
    seg[ivf&~now]          ="S4"                 # 현재시술용 아님
    seg[ivf&now&(tx==0)]   ="S3"                 # 이식=0
    seg[di]                ="S1"                 # DI
    return seg
seg=assign_segment(train).values
assert (~pd.Series(seg).isin(["S1","S2","S3","S4"])).sum()==0, "미분류 존재!"
print("세그먼트 분포 (목표 train: S1 2.5% / S2 83.3% / S3 9.4% / S4 4.9%):")
for s in ["S1","S2","S3","S4"]:
    m=(seg==s); print(f"  {s}: n={m.sum()} ({100*m.mean():.1f}%) 성공률={y[m].mean():.4f}")

세그먼트 분포 (목표 train: S1 2.5% / S2 83.3% / S3 9.4% / S4 4.9%):
  S1: n=6291 (2.5%) 성공률=0.1289
  S2: n=213516 (83.3%) 성공률=0.3062
  S3: n=24109 (9.4%) 성공률=0.0009
  S4: n=12435 (4.9%) 성공률=0.0006


## 2. 정본 tf_tree (train-only fit, 누수 안전) + 3모델 학습기

In [3]:
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}) if COL_PROC in tr else []
    return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    if COL_PROC in df:
        ts=df[COL_PROC].apply(_tp)
        for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if df[c].dtype==object]
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
st=fit_tree(train); X,CATF=tf_tree(train,st); CATIDX=[X.columns.get_loc(c) for c in CATF]
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50)
XGP=dict(objective="binary:logistic",eval_metric="auc",tree_method="hist",learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,verbosity=0)
def _ne(shallow): return NE_SHALLOW if shallow else NE_FULL
def fit3(Xtr,ytr,Xva,shallow,seed,Xcal=None):
    # 한 번 학습으로 Xva(+선택적 Xcal) 예측. Xcal은 calibration fit용 fold-train 자체예측.
    if len(np.unique(ytr))<2:                       # deterministic 세그먼트(전부 0 등)
        cst=float(ytr.mean()); pv=np.full(len(Xva),cst)
        return (pv, np.full(len(Xcal),cst)) if Xcal is not None else pv
    nl=15 if shallow else 63; md=4 if shallow else 6
    ml=lgb.train(dict(LGP,seed=seed,num_leaves=nl),lgb.Dataset(Xtr,ytr,categorical_feature=CATF),_ne(shallow))
    mx=xgb.train(dict(XGP,seed=seed,max_depth=md),xgb.DMatrix(Xtr.values,label=ytr),_ne(shallow))
    mc=CatBoostClassifier(iterations=_ne(shallow),learning_rate=0.05,depth=md,verbose=0,random_seed=seed); mc.fit(Xtr,ytr,cat_features=CATF)
    def _blend(Z): return (runr(ml.predict(Z))+runr(mx.predict(xgb.DMatrix(Z.values)))+runr(mc.predict_proba(Z)[:,1]))/3
    pv=_blend(Xva)
    return (pv, _blend(Xcal)) if Xcal is not None else pv
print("tf_tree",X.shape,"| cat",len(CATF),"| 3모델 학습기 준비")

tf_tree (256351, 72) | cat 5 | 3모델 학습기 준비


## 3. CELL2+3 — 공유 fold + 세그먼트별 3모델 OOF (멀티시드) + 무결성/누수 assert

In [4]:
from sklearn.isotonic import IsotonicRegression
def seg_oof(seed, calibrate=True):
    fold=np.full(N,-1)
    for f,(_,vi) in enumerate(StratifiedKFold(5,shuffle=True,random_state=seed).split(np.arange(N),y)): fold[vi]=f
    oof_raw=np.full(N,np.nan); oof_cal=np.full(N,np.nan)
    for s in ["S1","S2","S3","S4"]:
        sm=(seg==s); shallow=s in ("S1","S3","S4")
        for f in range(5):
            tr=sm&(fold!=f); va=sm&(fold==f)
            if va.sum()==0: continue
            # 한 번 학습으로 val 예측 + train 자체예측(calibration fit용). 누수 안전(cal fit=fold-train only)
            pva,ptr=fit3(X[tr],y[tr],X[va],shallow,seed,Xcal=X[tr])
            oof_raw[va]=pva
            if tr.sum()<20 or len(np.unique(y[tr]))<2:
                oof_cal[va]=float(y[tr].mean()) if tr.sum()>0 else 0.0
            else:
                ir=IsotonicRegression(out_of_bounds="clip").fit(ptr, y[tr])   # fold-train (예측,라벨)로 fit
                oof_cal[va]=ir.transform(pva)
    return (oof_cal if calibrate else oof_raw), oof_raw, fold
def global_oof(seed):   # 같은 base 피처, 비분리(구조 효과 격리용 baseline)
    fold=np.full(N,-1)
    for f,(_,vi) in enumerate(StratifiedKFold(5,shuffle=True,random_state=seed).split(np.arange(N),y)): fold[vi]=f
    o=np.full(N,np.nan)
    for f in range(5):
        tr=(fold!=f); va=(fold==f); o[va]=fit3(X[tr],y[tr],X[va],False,seed)
    return o
t0=time.time()
oof_seg,oof_raw,fold=seg_oof(42)
filled=~np.isnan(oof_seg)
assert filled.sum()>=N*0.999, f"OOF 미충전 {N-filled.sum()}행 — 세그먼트/fold 정합 확인!"
print(f"세그먼트 OOF 충전 {filled.sum()}/{N} ✅ ({(time.time()-t0)/60:.1f}분)")
for s in ["S1","S2","S3","S4"]:
    m=(seg==s)
    if m.sum()>100 and len(np.unique(y[m]))>1:
        a=roc_auc_score(y[m],oof_seg[m]); assert a<0.999, f"{s} 라벨 누수 의심!"; print(f"  {s}: n={m.sum()} OOF_AUC={a:.4f}")
    else: print(f"  {s}: n={m.sum()} (deterministic/소표본 — AUC 생략)")
# ★ 진단: 세그먼트별 raw pred 분포 (스케일 불일치 확인)
print("\n[진단] 세그먼트별 raw pred 분포 (S3 평균이 S2와 겹치면 스케일 불일치):")
for s in ["S1","S2","S3","S4"]:
    m=(seg==s); print(f"  {s}: 평균={oof_raw[m].mean():.4f} 범위=[{oof_raw[m].min():.4f},{oof_raw[m].max():.4f}] 성공률={y[m].mean():.4f}")
print(f"\nraw stitch 전체 AUC      ={roc_auc_score(y,oof_raw):.5f}  (스케일 안 맞으면 부분 최솟값보다↓)")
print(f"세그먼트별 calibration 후 ={roc_auc_score(y,oof_seg):.5f}  ← 이게 정당한 세그먼트 OOF (게이트에 사용)")

세그먼트 OOF 충전 256351/256351 ✅ (22.5분)
  S1: n=6291 OOF_AUC=0.6684
  S2: n=213516 OOF_AUC=0.6687
  S3: n=24109 OOF_AUC=0.5059
  S4: n=12435 OOF_AUC=0.4993

[진단] 세그먼트별 raw pred 분포 (S3 평균이 S2와 겹치면 스케일 불일치):
  S1: 평균=0.5004 범위=[0.0007,0.9995] 성공률=0.1289
  S2: 평균=0.5000 범위=[0.0000,1.0000] 성공률=0.3062
  S3: 평균=0.5001 범위=[0.0064,0.9995] 성공률=0.0009
  S4: 평균=0.5002 범위=[0.0023,1.0000] 성공률=0.0006

raw stitch 전체 AUC      =0.66153  (스케일 안 맞으면 부분 최솟값보다↓)
세그먼트별 calibration 후 =0.73502  ← 이게 정당한 세그먼트 OOF (게이트에 사용)


## 4. CELL4 — 게이트 ① 구조 격리(vs 비분리) ② 운영 best(0.74219) 대비

In [5]:
def hill(d,yy,n=120):  # 0.74219 조합 직접 힐클라이밍 재구성 (근사 가중치 추측 금지)
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; cnt=Counter(best[0]); w={k:cnt.get(k,0)/len(best[0]) for k in nm}
    return w, sum(w[k]*d[k] for k in nm)
def paired_ci(pa,pb,n=2000,seed=0):
    r=np.random.default_rng(seed); ds=np.empty(n)
    for i in range(n):
        ix=r.integers(0,N,N); ds[i]=roc_auc_score(y[ix],pa[ix])-roc_auc_score(y[ix],pb[ix])
    return float(ds.mean()),float(np.percentile(ds,2.5)),float(np.percentile(ds,97.5))
# ── 게이트① : 세그먼트 vs 같은 base 비분리 글로벌 (구조 효과만 격리)
gl=global_oof(42); a_seg=roc_auc_score(y,oof_seg); a_gl=roc_auc_score(y,gl)
inc1,lo1,hi1=paired_ci(oof_seg,gl)
print(f"[게이트①] 세그먼트 {a_seg:.5f} vs 비분리 글로벌 {a_gl:.5f} | 증분 {inc1:+.5f} CI[{lo1:+.5f},{hi1:+.5f}] {'✅ 구조 효과' if lo1>0 else '— 구조 중립'}")
# ── 게이트② : 세그먼트 vs 운영 best(0.74219) — 멤버 OOF 있으면 힐클라이밍 재구성
def try_operational():
    try:
        op=pd.read_csv(P("oof_predictions.csv")); v2=pd.read_csv(P("oof_v2_trees.csv"))
        v3=pd.read_csv(P("oof_v3_trees.csv")); v23=pd.read_csv(P("oof_v2v3_trees.csv"))
        lr=pd.read_csv(P("oof_lin_ratio.csv")); nn=pd.read_csv(P("oof_nn.csv"))
        for d in [op,v2,v3,v23,lr,nn]:
            if len(d)!=N: return None   # DRYRUN 서브샘플 길이 불일치 → 스킵
        # 0.74219 FORCED 조합: lgb:v2v3, xgb:v3, cat:v2v3, lin:ratio(=v2), nn:base
        members={"lgb":runr(v23["oof_v2v3_lgb"].values),"xgb":runr(v3["oof_v3_xgb"].values),
                 "cat":runr(v23["oof_v2v3_cat"].values),"lin":runr(lr["oof_lin_ratio"].values),
                 "nn":runr(nn["oof_nn"].values)}
        w,p=hill(members,y)
        print(f"  운영best(0.74219) 재구성 AUC={roc_auc_score(y,p):.5f} | 가중치={ {k:round(v,2) for k,v in w.items() if v>0} }")
        return p
    except Exception as e:
        print("  운영best 재구성 불가:",e); return None
p_best=try_operational()
if p_best is not None:
    a_best=roc_auc_score(y,p_best); inc2,lo2,hi2=paired_ci(oof_seg,p_best)
    print(f"[게이트②] 세그먼트 {a_seg:.5f} vs 운영best {a_best:.5f} | 증분 {inc2:+.5f} CI[{lo2:+.5f},{hi2:+.5f}] {'✅ 운영 초과' if lo2>0 else '— 운영 못넘음'}")
else:
    print("[게이트②] 운영best OOF 재구성 스킵(풀런·멤버 입력 시 활성). 게이트①로 구조 효과만 판정.")
print("\n해석: ①+ & CI0제외 → 구조 레버(LB 1발). ①중립 → 구조 단독 중립, 이득은 세그먼트별 튜닝(발표 '향후').")

[게이트①] 세그먼트 0.73502 vs 비분리 글로벌 0.73627 | 증분 -0.00125 CI[-0.00155,-0.00094] — 구조 중립
  운영best(0.74219) 재구성 AUC=0.74067 | 가중치={'lgb': 0.28, 'xgb': 0.22, 'cat': 0.26, 'lin': 0.03, 'nn': 0.21}
[게이트②] 세그먼트 0.73502 vs 운영best 0.74067 | 증분 -0.00564 CI[-0.00626,-0.00504] — 운영 못넘음

해석: ①+ & CI0제외 → 구조 레버(LB 1발). ①중립 → 구조 단독 중립, 이득은 세그먼트별 튜닝(발표 '향후').


## 5. 멀티시드 재확인 (DRYRUN=False & SEEDS 다중일 때) + 저장

In [6]:
if len(SEEDS)>1:
    incs=[]
    for s in SEEDS:
        os_,_,_=seg_oof(s); gl_=global_oof(s); incs.append(roc_auc_score(y,os_)-roc_auc_score(y,gl_))
    import numpy as _np
    print(f"멀티시드 구조 증분(vs 비분리): {[round(i,5) for i in incs]} | 평균 {_np.mean(incs):+.5f}±{_np.std(incs):.5f}",
          "→ ✅ 시드 견고" if (_np.mean(incs)>0 and abs(_np.mean(incs))>_np.std(incs)) else "→ 노이즈")
else:
    print("멀티시드: DRYRUN=False로 SEEDS=(42,7,2024) 풀런 시 활성")
outdir="/kaggle/working" if os.path.isdir("/kaggle/working") else "."
pd.DataFrame({"oof_segment":oof_seg,"y":y}).to_csv(f"{outdir}/oof_segment.csv",index=False)
print(f"💾 {outdir}/oof_segment.csv (그리드 v3 best와 ρ 상관 → 중복/보완)")

멀티시드 구조 증분(vs 비분리): [np.float64(-0.00124), np.float64(-0.00155), np.float64(-0.00124)] | 평균 -0.00135±0.00015 → 노이즈
💾 /kaggle/working/oof_segment.csv (그리드 v3 best와 ρ 상관 → 중복/보완)
